## Define required widgets

In [ ]:
dbutils.widgets.text("format", "")
dbutils.widgets.text("header", "")
dbutils.widgets.text("inferColumnTypes", "")
dbutils.widgets.text("schemaLocation", "")
dbutils.widgets.text("checkpoint", "")
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("outputMode", "")
dbutils.widgets.text("mergeSchema", "")
dbutils.widgets.text("emc_target", "")

## Get the widget values

In [ ]:
format = dbutils.widgets.get("format")
header = dbutils.widgets.get("header")
inferColumnTypes = dbutils.widgets.get("inferColumnTypes")
schemaLocation = dbutils.widgets.get("schemaLocation")
checkpoint = dbutils.widgets.get("checkpoint")
source_path = dbutils.widgets.get("source_path")
outputMode = dbutils.widgets.get("outputMode")
mergeSchema = dbutils.widgets.get("mergeSchema")
emc_target = dbutils.widgets.get("emc_target")

In [ ]:
# Start autoloader readstream files
df = (
    spark.readStream
    .format("cloudFiles")  # Auto Loader format
    .option("cloudFiles.format", format)  # Source file format (csv, json, parquet)
    .option("header", header)
    .option("cloudFiles.inferColumnTypes", inferColumnTypes)
    .option("cloudFiles.schemaLocation", schemaLocation)  # Schema inference location
    .load(source_path)
)
                
# Start the streaming write
query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint)  # Required for streaming
    .outputMode(outputMode)                    # e.g., "append" or "complete"
    .option("mergeSchema", mergeSchema)
    .trigger(availableNow=True)
    .toTable(emc_target)                       # Table name in the metastore
)

# Wait for the stream to finish
query.awaitTermination()